In [ ]:
pip install pandas numpy matplotlib seaborn scipy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar datos
df = pd.read_csv("amz_uk_price_prediction_dataset.csv")

# Tabla de frecuencia por categoría
category_counts = df['category'].value_counts()
print("Top 5 categorías más frecuentes:\n", category_counts.head(5))

# Bar chart top 10 categorías
category_counts.head(10).plot(kind='bar', title="Top 10 Product Categories")
plt.show()

# Pie chart top 5 categorías
category_counts.head(5).plot(kind='pie', autopct='%1.1f%%', title="Proportion of Top 5 Categories")
plt.ylabel("")
plt.show()


In [ ]:
# Medidas de tendencia central para precios
print("Price - Mean:", df['price'].mean())
print("Price - Median:", df['price'].median())
print("Price - Mode:", df['price'].mode()[0])

# Medidas de dispersión
print("Price - Std:", df['price'].std())
print("Price - IQR:", df['price'].quantile(0.75) - df['price'].quantile(0.25))

# Histograma y boxplot
df['price'].plot(kind='hist', bins=50, title="Distribution of Product Prices")
plt.show()

df['price'].plot(kind='box', title="Boxplot of Product Prices")
plt.show()

# Medidas para ratings
print("Rating - Mean:", df['stars'].mean())
print("Rating - Median:", df['stars'].median())
print("Rating - Mode:", df['stars'].mode()[0])

# Histograma y boxplot ratings
df['stars'].plot(kind='hist', bins=20, title="Distribution of Product Ratings")
plt.show()

df['stars'].plot(kind='box', title="Boxplot of Product Ratings")
plt.show()


In [ ]:
# -------------------------------
# Lab: EDA Bivariate Analysis
# -------------------------------

# 1️⃣ Instalación de librerías necesarias (si no están)
# pip install pandas matplotlib seaborn numpy

# 2️⃣ Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
plt.style.use("default")
plt.rcParams["figure.figsize"] = (10,6)

# -------------------------------
# Cargar datos
# -------------------------------
df = pd.read_csv("amz_uk_price_prediction_dataset.csv")

# Convertir columnas a numéricas si es necesario
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['stars'] = pd.to_numeric(df['stars'], errors='coerce')

# Eliminar filas con valores faltantes en columnas relevantes
df = df.dropna(subset=['price', 'stars', 'category'])

# ================================
# Part 1: Analyzing Best-Seller Trends Across Product Categories
# ================================

# Frequency Table
category_counts = df['category'].value_counts()
print("Top 5 Categories:\n", category_counts.head(5))

# Bar chart top 10 categories
category_counts.head(10).plot(kind='bar', title="Top 10 Product Categories")
plt.show()

# Pie chart top 5 categories
category_counts.head(5).plot(kind='pie', autopct='%1.1f%%', title="Proportion of Top 5 Categories")
plt.ylabel("")
plt.show()

# ================================
# Part 2: Preliminary Step - Remove outliers in product prices
# ================================

Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

# Mantener solo valores dentro del rango 1.5*IQR
df_clean = df[(df['price'] >= Q1 - 1.5*IQR) & (df['price'] <= Q3 + 1.5*IQR)]

# ================================
# Part 2: Violin Plots for Price Distribution Across Categories
# ================================
top5_categories = df_clean['category'].value_counts().head(5).index
sns.violinplot(x='category', y='price', data=df_clean[df_clean['category'].isin(top5_categories)])
plt.title("Price Distribution Across Top 5 Categories (Violin Plot)")
plt.show()

# ================================
# Part 2: Bar Charts for Average Prices in Top Categories
# ================================
avg_prices = df_clean.groupby('category')['price'].mean().sort_values(ascending=False)
avg_prices.head(10).plot(kind='bar', title="Average Price in Top 10 Categories")
plt.ylabel("Average Price (£)")
plt.show()

# ================================
# Part 2: Box Plots for Ratings Distribution by Category
# ================================
sns.boxplot(x='category', y='stars', data=df[df['category'].isin(top5_categories)])
plt.title("Ratings Distribution Across Top 5 Categories (Box Plot)")
plt.show()

# ================================
# Part 3: Investigating Price vs Ratings
# ================================
sns.scatterplot(x='price', y='stars', data=df_clean, alpha=0.5)
plt.title("Price vs Ratings")
plt.xlabel("Price (£)")
plt.ylabel("Stars")
plt.show()

print("Correlation between price and rating:", df_clean['price'].corr(df_clean['stars']))


In [ ]:
# -------------------------------
# Part 1 - Crosstab Analysis entre category e isBestSeller
# -------------------------------

# Asegurarse de que la columna isBestSeller exista y sea categórica
df['isBestSeller'] = df['isBestSeller'].astype(str)

# Crosstab: frecuencia de cada combinación
crosstab = pd.crosstab(df['category'], df['isBestSeller'])
print("Crosstab category x isBestSeller:\n", crosstab)

# -------------------------------
# Part 1 - Pruebas estadísticas
# -------------------------------

from scipy.stats import chi2_contingency

# Chi-cuadrado de independencia
chi2, p, dof, ex = chi2_contingency(crosstab)
print("\nChi-square test:")
print("Chi2:", chi2, "p-value:", p)

# Cramér's V para fuerza de asociación
def cramers_v(confusion_matrix):
    chi2, _, _, _ = chi2_contingency(confusion_matrix)
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return (chi2 / (n * (min(r-1, k-1))))**0.5

cramers_v_value = cramers_v(crosstab)
print("Cramér's V:", cramers_v_value)

# -------------------------------
# Part 1 - Visualización: Stacked Bar Chart
# -------------------------------

crosstab_norm = crosstab.div(crosstab.sum(axis=1), axis=0)  # normalizar por fila
crosstab_norm.plot(kind='bar', stacked=True, colormap='Set2')
plt.title("Stacked Bar Chart: Category vs isBestSeller")
plt.ylabel("Proportion")
plt.xlabel("Category")
plt.xticks(rotation=45, ha='right')
plt.legend(title='isBestSeller')
plt.show()


In [ ]:
# Scatter plot precio vs rating
plt.scatter(df['price'], df['stars'], alpha=0.5)
plt.title("Price vs Rating")
plt.xlabel("Price (£)")
plt.ylabel("Stars")
plt.show()

# Correlación
print("Correlation between price and rating:", df['price'].corr(df['stars']))


In [ ]:
# -------------------------------
# Part 3 - Correlation Heatmap de variables numéricas
# -------------------------------

# Seleccionamos solo columnas numéricas relevantes
numeric_cols = ['price', 'stars', 'reviews']  # agrega otras si existen
numeric_df = df[numeric_cols]

# Heatmap de correlación
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap of Numeric Variables")
plt.show()

# -------------------------------
# Part 3 - QQ plot para comprobar normalidad del precio
# -------------------------------
import scipy.stats as stats

plt.figure(figsize=(6,6))
stats.probplot(df['price'], dist="norm", plot=plt)
plt.title("QQ Plot for Price")
plt.show()

# -------------------------------
# Bonus: análisis repitiendo sin eliminar outliers
# -------------------------------

# Si antes habías filtrado outliers en price, aquí usamos todo el dataset original
numeric_df_full = df[numeric_cols]  # sin filtrar
corr_full = numeric_df_full.corr()

sns.heatmap(corr_full, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap (Full Data, Including Outliers)")
plt.show()

plt.figure(figsize=(6,6))
stats.probplot(df['price'], dist="norm", plot=plt)
plt.title("QQ Plot for Price (Full Data, Including Outliers)")
plt.show()
